# 02. Feature checks

Sanity checks on the modelling panel and a demonstration of the leakage control.

In [1]:
import numpy as np
import pandas as pd
from newsgate.config import load_config
from newsgate.pipeline import RunPaths

cfg = load_config('configs/synthetic.yaml')
panel = pd.read_parquet(RunPaths.from_config(cfg).root / 'panel.parquet')
panel.shape

(32160, 52)

In [2]:
feature_cols = [c for c in panel.columns if c.split('_')[0] in {'px', 'att', 'sent', 'evt'}]
panel[feature_cols].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
px_ret_0,32160.0,0.000,0.025,-0.209,-0.013,0.000,0.014,0.218
px_ret_1,32160.0,0.000,0.025,-0.209,-0.013,0.000,0.014,0.218
px_ret_2,32160.0,0.000,0.025,-0.209,-0.013,0.000,0.014,0.218
px_ret_3,32160.0,0.000,0.025,-0.209,-0.013,0.000,0.014,0.218
px_ret_4,32160.0,0.000,0.025,-0.209,-0.013,0.000,0.014,0.218
px_mom_5,32160.0,0.001,0.055,-0.613,-0.028,0.001,0.031,0.470
px_mom_10,32160.0,0.002,0.078,-0.719,-0.040,0.003,0.045,0.548
px_mom_20,32160.0,0.005,0.109,-0.847,-0.055,0.007,0.066,1.043
px_mom_60,32160.0,0.014,0.189,-1.006,-0.098,0.017,0.125,0.917
px_vol_5,32160.0,0.021,0.012,0.001,0.013,0.019,0.027,0.124


## Missing values and extreme values

In [3]:
panel[feature_cols].isna().mean().sort_values(ascending=False).head(10)

px_ret_0     0.0
px_ret_1     0.0
px_ret_2     0.0
px_ret_3     0.0
px_ret_4     0.0
px_mom_5     0.0
px_mom_10    0.0
px_mom_20    0.0
px_mom_60    0.0
px_vol_5     0.0
dtype: float64

## Which features line up with tomorrow's absolute move? (association only)

In [4]:
panel[feature_cols + ['tgt_abs']].corr(method='spearman')['tgt_abs'].drop('tgt_abs').sort_values(key=abs, ascending=False).head(15).round(3)

px_vol_20           0.284
px_vol_5            0.249
px_vol_60           0.232
px_range            0.146
px_absret_0         0.141
px_mkt_vol_20       0.076
att_surprise        0.039
att_n               0.032
att_log_n           0.032
evt_n_events        0.031
evt_max_cluster     0.027
sent_std            0.024
att_has_news        0.022
att_days_since     -0.021
evt_repeat_share    0.018
Name: tgt_abs, dtype: float64

## Leakage control

Features for a date must not change when later data is removed. `tests/test_features_leakage.py` checks this on every run; this cell repeats the idea by hand.

In [5]:
from newsgate.data import build_sources
from newsgate.features.panel import build_panel

sources = build_sources(cfg.data)
news, prices = sources.news.load(), sources.prices.load(sources.tickers)
cut = pd.Timestamp(np.sort(panel['date'].unique())[len(panel['date'].unique()) // 2])
short, _ = build_panel(cfg, news[news['published_at'] < cut + pd.Timedelta(days=1)], prices[prices['date'] <= cut], sources.tickers)
key = ['ticker', 'date']
a = panel[panel['date'] <= cut].set_index(key)[feature_cols].sort_index()
b = short[short['date'] <= cut].set_index(key)[feature_cols].sort_index()
print('max absolute difference:', float((a - b).abs().max().max()))

max absolute difference: 0.0
